# Planes, segments, boxes, rays

Curb faces are often vertical. A fit of the form `z = ax + by + c` cannot represent them. Use a general plane, and keep a tolerance instead of comparing floats with `==`.


These cells are PyTorch. Geometry wants float64, so the setup cell calls `torch.set_default_dtype(torch.float64)`. The library default is float32, and a plane SVD or a Kabsch in float32 will miss a hand check that passes in float64.


In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Fit a plane and measure distance

Fit one plane to every point. Return the centroid and a unit normal. Also write the signed distance from arbitrary points to that plane, positive on the normal's side.

Use a general plane `ax + by + cz = d`. Flip the normal so its largest component is positive; the SVD sign is arbitrary, and a stable sign makes tests possible.

The fit uses every point. One outlier will tilt it. That is what the RANSAC problem is for.

Hand check: points with `x = 0` and varying `y, z` have normal `(1, 0, 0)`, not a normal with a large `z` component.

<details><summary>Hint</summary>

Center the points. `_, _, Vh = torch.linalg.svd(centered, full_matrices=False)`. The normal is `Vh[-1]`, the direction of least variance. Signed distance is `(points - centroid) @ unit_normal`.

</details>


In [ ]:
import torch

def fit_plane(points):
    """
    Least-squares plane through all points.

    Returns centroid (3,) and a unit normal (3,).
    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    raise NotImplementedError

def signed_plane_distances(points, centroid, normal):
    """
    Signed distance from each point to the plane. Positive along `normal`.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def fit_plane(points):
    """Least-squares plane. Returns centroid and a unit normal.

    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    points = _t(points)
    if points.shape[0] < 3:
        raise ValueError("need at least 3 points")
    centroid = points.mean(dim=0)
    _, _, Vh = torch.linalg.svd(points - centroid, full_matrices=False)
    normal = Vh[-1].clone()
    k = int(torch.argmax(normal.abs()))
    if normal[k] < 0.0:
        normal = -normal
    return centroid, normal

def signed_plane_distances(points, centroid, normal):
    """Signed distance from each point to the plane. Positive along `normal`."""
    points = _t(points)
    normal = _t(normal)
    n = torch.linalg.norm(normal)
    if n < 1e-12:
        raise ValueError("normal has zero length")
    unit = normal / n
    return (points - _t(centroid)) @ unit

def point_plane_loss(points, centroid, normal):
    """Mean squared signed distance. Differentiable in `points`."""
    return signed_plane_distances(points, centroid, normal).square().mean()

def check_fit_plane():
    g = torch.Generator().manual_seed(0)
    xy = torch.randn(30, 2, generator=g, dtype=DTYPE)
    horizontal = torch.column_stack((xy, 0.01 * torch.randn(30, generator=g, dtype=DTYPE)))
    c, n = fit_plane(horizontal)
    assert torch.allclose(n, torch.tensor([0.0, 0.0, 1.0]), atol=0.05)
    assert signed_plane_distances(horizontal, c, n).abs().amax() < 0.05
    g = torch.Generator().manual_seed(0)
    vertical = torch.column_stack(
        (
            torch.zeros(20, dtype=DTYPE),
            torch.randn(20, generator=g, dtype=DTYPE),
            torch.randn(20, generator=g, dtype=DTYPE),
        )
    )
    _, n = fit_plane(vertical)
    assert torch.allclose(n.abs(), torch.tensor([1.0, 0.0, 0.0]), atol=1e-6)
    assert n[0] > 0
    pts = torch.randn(12, 3, dtype=DTYPE, requires_grad=True)
    point_plane_loss(pts, pts.detach().mean(0), torch.tensor([0.0, 0.0, 1.0])).backward()
    assert pts.grad is not None and torch.isfinite(pts.grad).all()
    print("passed")

check_fit_plane()


## 2. Point-to-segment distance

For many points and one finite segment `ab`, return the Euclidean distance to the segment and the closest point on the segment.

Clamp the segment parameter to `[0, 1]`. If you leave it unclamped, you have distance to the infinite line, which is a different question and worth naming if they ask. A zero-length segment is just distance to `a`.

Hand check: segment `(0,0,0)`–`(1,0,0)`. The point `(0.5, 1, 0)` is at distance 1, closest `(0.5, 0, 0)`. The point `(2, 0, 0)` is at distance 1, closest `(1, 0, 0)`.


In [ ]:
import torch

def point_to_segment(points, a, b):
    """
    Distance from each point to the finite segment ab.

    points: (N, D), a and b: (D,)
    returns distances (N,), closest points (N, D).
    `t` is clamped to [0, 1]. Leave it unclamped for an infinite line.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def point_to_segment(points, a, b):
    """Distance from each point to the finite segment ab.

    returns distances (N,), closest points (N, D).
    """
    points = _t(points)
    a = _t(a)
    b = _t(b)
    ab = b - a
    denom = torch.dot(ab, ab)
    if denom < 1e-24:
        closest = a.expand_as(points).clone()
        return torch.linalg.norm(points - a, dim=1), closest
    t = torch.clamp(((points - a) @ ab) / denom, 0.0, 1.0)
    closest = a + t[:, None] * ab
    return torch.linalg.norm(points - closest, dim=1), closest

def check_point_to_segment():
    pts = torch.tensor([[0.5, 1.0, 0.0], [0.0, 1.0, 0.0], [2.0, 0.0, 0.0], [0.2, 0.2, 0.0]])
    dist, closest = point_to_segment(pts, torch.tensor([0.0, 0.0, 0.0]), torch.tensor([1.0, 0.0, 0.0]))
    assert torch.allclose(dist, torch.tensor([1.0, 1.0, 1.0, 0.2]))
    assert torch.allclose(closest[0], torch.tensor([0.5, 0.0, 0.0]))
    assert torch.allclose(closest[2], torch.tensor([1.0, 0.0, 0.0]))
    dist, _ = point_to_segment(pts[:1], torch.ones(3), torch.ones(3))
    assert torch.allclose(dist, torch.linalg.norm(pts[0] - 1))
    print("passed")

check_point_to_segment()


## 3. 3D axis-aligned box IoU

Intersection over union of two axis-aligned boxes. Each box is a min corner and a max corner, shape `(3,)`.

Intersect by clipping each axis independently, then multiply the three positive extents. A shared face has volume 0, so IoU 0.

Hand check: `[0,0,0]–[2,2,2]` against `[1,1,1]–[3,3,3]` has intersection volume 1 and union 15, so IoU `1/15`.


In [ ]:
import torch

def aabb_iou(a_min, a_max, b_min, b_max):
    """
    Intersection-over-union of two axis-aligned boxes.

    Each argument is (D,). Works in 2D or 3D. Touching faces have IoU 0
    because the intersection volume is zero.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def aabb_iou(a_min, a_max, b_min, b_max):
    """Intersection-over-union of two axis-aligned boxes."""
    a_min, a_max = _t(a_min), _t(a_max)
    b_min, b_max = _t(b_min), _t(b_max)
    inter_ext = torch.clamp(torch.minimum(a_max, b_max) - torch.maximum(a_min, b_min), min=0.0)
    inter = torch.prod(inter_ext)
    union = torch.prod(a_max - a_min) + torch.prod(b_max - b_min) - inter
    if union <= 0:
        return inter * 0.0
    return inter / union

def check_aabb_iou():
    iou = aabb_iou([0, 0, 0], [2, 2, 2], [1, 1, 1], [3, 3, 3])
    assert torch.allclose(iou, torch.tensor(1.0 / 15.0))
    assert float(aabb_iou([0, 0, 0], [1, 1, 1], [0, 0, 0], [1, 1, 1])) == 1.0
    assert float(aabb_iou([0, 0, 0], [1, 1, 1], [2, 2, 2], [3, 3, 3])) == 0.0
    assert float(aabb_iou([0, 0, 0], [1, 1, 1], [1, 1, 1], [2, 2, 2])) == 0.0
    print("passed")

check_aabb_iou()


## 4. Points inside an oriented box

An oriented box is a center, a rotation, and per-axis half extents. `R` maps local coordinates to world coordinates: its columns are the box axes in the world. A world point `p` has local coordinates `R.T @ (p - center)`.

Return a boolean mask. Treat the boundary as inside, with a small tolerance so a point on a face is not rejected by rounding.

Hand check: `R` is a `+90°` rotation about Z,

```
[[0, -1, 0],
 [1,  0, 0],
 [0,  0, 1]]
```

half extents `(2, 0.5, 1)`, center at the origin. The world point `(0, 1.5, 0)` is inside. The world point `(1.5, 0, 0)` is outside. Getting those swapped means the local transform is transposed.

<details><summary>Hint</summary>

For row points, `local = (points - center) @ R`.

</details>


In [ ]:
import torch

def points_in_obb(points, center, R, half_extents, tol=1e-08):
    """
    Which points lie in an oriented box.

    World point p corresponds to local = R.T @ (p - center), and R's columns
    are the box axes expressed in world coordinates. A point is inside when
    every |local_i| <= half_extents[i].
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def points_in_obb(points, center, R, half_extents, tol=1e-8):
    """Which points lie in an oriented box.

    R's columns are the box axes in world coordinates.
    Local coordinates of a row point are (p - center) @ R.
    """
    points = _t(points)
    center = _t(center)
    half_extents = _t(half_extents)
    local = (points - center) @ R
    return (local.abs() <= half_extents + tol).all(dim=1)

def check_points_in_obb():
    R = torch.tensor([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
    pts = torch.tensor([[0.0, 1.5, 0.0], [1.5, 0.0, 0.0], [0.0, 0.0, 0.9], [0.0, 0.0, 1.2]])
    mask = points_in_obb(pts, torch.zeros(3), R, torch.tensor([2.0, 0.5, 1.0]))
    assert torch.equal(mask, torch.tensor([True, False, True, False]))
    print("passed")

check_points_in_obb()


## 5. Ray-triangle intersection

Intersect many rays with one triangle using Möller–Trumbore. `directions` are not necessarily unit length; `t` is the scale in `origin + t * direction`.

Count a hit when `t > 0` and the barycentric coordinates lie inside the triangle, including the boundary. A parallel ray misses. A ray that starts on the triangle and points away misses. `t` only needs to be meaningful where the hit mask is true.

Hand check: triangle `(0,0,0)`, `(1,0,0)`, `(0,1,0)`.

- origin `(0.25, 0.25, 1)`, direction `(0, 0, -1)` hits at `t = 1`
- origin `(0, 0, 2)`, direction `(0, 0, -1)` hits the vertex, `t = 2`
- origin `(0.5, 0.5, 1)`, direction `(0, 0, -1)` hits the hypotenuse
- origin `(2, 2, 1)` misses
- origin `(0.25, 0.25, -1)`, direction `(0, 0, -1)` misses (`t` would be negative)
- a ray lying in the plane misses

<details><summary>Hint</summary>

`h = dir × edge2`, `a = edge1 · h`. Skip `|a| < eps`. Then `u`, `v`, and `t` are the usual three scaled dots. Guard the division so a parallel ray does not warn.

`torch.linalg.cross` does not broadcast a batch of directions `(N, 3)` against one edge `(3,)`. Broadcast the two arguments to the same shape first.

</details>


In [ ]:
import torch

def ray_triangle(origins, directions, v0, v1, v2, eps=1e-08):
    """
    Möller–Trumbore intersection of many rays with one triangle.

    origins, directions: (N, 3). `t` solves origin + t * direction, and the
    direction does not need to be a unit vector.
    A hit requires t > eps (in front of the origin) and barycentric coordinates
    inside the triangle, including the boundary.
    Parallel rays miss. `t` is only meaningful where hit is True.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _cross(a, b):
    a, b = torch.broadcast_tensors(a, b)
    return torch.linalg.cross(a, b, dim=-1)

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def ray_triangle(origins, directions, v0, v1, v2, eps=1e-8):
    """Möller–Trumbore. Many rays, one triangle.

    returns t (N,), hit (N,). t is meaningful where hit is True.
    """
    origins = _t(origins)
    directions = _t(directions)
    v0, v1, v2 = _t(v0), _t(v1), _t(v2)
    edge1 = v1 - v0
    edge2 = v2 - v0
    h = _cross(directions, edge2)
    a = h @ edge1
    parallel = a.abs() < eps
    f = torch.zeros_like(a)
    f = torch.where(parallel, f, 1.0 / a)
    s = origins - v0
    u = f * (s * h).sum(dim=1)
    q = _cross(s, edge1)
    v = f * (directions * q).sum(dim=1)
    t = f * (q @ edge2)
    hit = (~parallel) & (u >= 0.0) & (v >= 0.0) & (u + v <= 1.0) & (t > eps)
    return t, hit

def check_ray_triangle():
    v0 = torch.zeros(3)
    v1 = torch.tensor([1.0, 0.0, 0.0])
    v2 = torch.tensor([0.0, 1.0, 0.0])
    origins = torch.tensor(
        [
            [0.25, 0.25, 1.0],
            [0.0, 0.0, 2.0],
            [0.5, 0.5, 1.0],
            [2.0, 2.0, 1.0],
            [0.25, 0.25, -1.0],
            [0.2, 0.2, 0.0],
        ]
    )
    directions = torch.tensor(
        [
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [1.0, 0.0, 0.0],
        ]
    )
    t, hit = ray_triangle(origins, directions, v0, v1, v2)
    assert torch.equal(hit, torch.tensor([True, True, True, False, False, False]))
    assert torch.allclose(t[0], torch.tensor(1.0))
    assert torch.allclose(origins[0] + t[0] * directions[0], torch.tensor([0.25, 0.25, 0.0]))
    assert torch.allclose(t[1], torch.tensor(2.0))
    print("passed")

check_ray_triangle()


## 6. RANSAC plane

Fit a plane when some points do not belong to it. Each hypothesis is 3 points. Skip a collinear sample. Score a hypothesis by how many points fall within `threshold` of its plane. After the loop, refit a least-squares plane on the inliers and recompute the mask from that refit plane.

A Python loop over hypotheses is the right shape. The distance test inside a hypothesis should be vectorized. Use `torch.Generator().manual_seed(seed)` and `torch.randperm(..., generator=g)` so a given seed is reproducible.

The normal follows the same sign rule as the least-squares fit: largest component positive.

Hand check: a set of points on `z = 0` plus a few points at `z ≈ 5`, threshold `0.05`. The normal is `(0,0,1)` and the high points are outliers. Repeat the idea on the plane `x = 0` with outliers at `x = 5`. A vertical inlier set is the one that fails if you parameterized the plane as a height field.

The reference cell includes the least-squares helpers so it runs alone.


In [ ]:
import torch

def ransac_plane(points, threshold, iterations, seed=0):
    """
    Fit a plane by RANSAC, then refit the plane on the inliers.

    Returns centroid, unit normal (largest component positive), and a boolean
    inlier mask measured against the refit plane.
    Hypotheses that are collinear are skipped. A Python loop over iterations
    is the expected shape; each trial itself is vectorized over the points.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _cross(a, b):
    a, b = torch.broadcast_tensors(a, b)
    return torch.linalg.cross(a, b, dim=-1)

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def fit_plane(points):
    """Least-squares plane. Returns centroid and a unit normal.

    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    points = _t(points)
    if points.shape[0] < 3:
        raise ValueError("need at least 3 points")
    centroid = points.mean(dim=0)
    _, _, Vh = torch.linalg.svd(points - centroid, full_matrices=False)
    normal = Vh[-1].clone()
    k = int(torch.argmax(normal.abs()))
    if normal[k] < 0.0:
        normal = -normal
    return centroid, normal

def signed_plane_distances(points, centroid, normal):
    """Signed distance from each point to the plane. Positive along `normal`."""
    points = _t(points)
    normal = _t(normal)
    n = torch.linalg.norm(normal)
    if n < 1e-12:
        raise ValueError("normal has zero length")
    unit = normal / n
    return (points - _t(centroid)) @ unit

def ransac_plane(points, threshold, iterations, seed=0):
    """RANSAC plane, then a least-squares refit on the inliers.

    Returns centroid, unit normal, and a boolean inlier mask.
    """
    points = _t(points)
    if points.shape[0] < 3:
        raise ValueError("need at least 3 points")
    g = torch.Generator()
    g.manual_seed(seed)
    best_inliers = None
    best_count = -1
    n = points.shape[0]
    for _ in range(iterations):
        choice = torch.randperm(n, generator=g)[:3]
        a, b, c = points[choice]
        normal = _cross(b - a, c - a)
        length = torch.linalg.norm(normal)
        if length < 1e-12:
            continue
        normal = normal / length
        dist = ((points - a) @ normal).abs()
        inliers = dist <= threshold
        count = int(inliers.sum())
        if count > best_count:
            best_count = count
            best_inliers = inliers
    if best_inliers is None or best_count < 3:
        raise RuntimeError("no valid plane hypothesis")
    centroid, normal = fit_plane(points[best_inliers])
    inliers = signed_plane_distances(points, centroid, normal).abs() <= threshold
    if int(inliers.sum()) >= 3:
        centroid, normal = fit_plane(points[inliers])
        inliers = signed_plane_distances(points, centroid, normal).abs() <= threshold
    return centroid, normal, inliers

def check_ransac_plane():
    ground = torch.column_stack(
        (torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20), torch.zeros(20))
    )
    extra = torch.tensor([[0.2, -0.4, 0.0], [-0.7, 0.3, 0.0], [0.5, 0.5, 0.0]])
    outliers = torch.tensor([[0.0, 0.0, 5.0], [0.4, -0.2, 4.0], [-0.3, 0.6, 6.0]])
    pts = torch.cat((ground, extra, outliers), dim=0)
    _, normal, mask = ransac_plane(pts, threshold=0.05, iterations=40, seed=0)
    assert torch.allclose(normal, torch.tensor([0.0, 0.0, 1.0]), atol=1e-6)
    assert int(mask.sum()) == pts.shape[0] - outliers.shape[0]
    assert not bool(mask[-outliers.shape[0] :].any())

    g = torch.Generator().manual_seed(1)
    y = torch.empty(30).uniform_(-2, 2, generator=g)
    z = torch.empty(30).uniform_(-2, 2, generator=g)
    wall = torch.column_stack((torch.zeros(30), y, z))
    bad = torch.empty(10, 3).uniform_(-2, 2, generator=g)
    bad[:, 0] = 5.0
    pts = torch.cat((wall, bad), dim=0)
    _, normal, mask = ransac_plane(pts, threshold=0.05, iterations=80, seed=1)
    assert torch.allclose(normal, torch.tensor([1.0, 0.0, 0.0]), atol=1e-6)
    assert int(mask.sum()) == 30
    print("passed")

check_ransac_plane()
